# TF-IDF Test Evaluation and Error Analysis

In this notebook, we evaluate the saved TF-IDF + PyTorch
logistic-regression model on the official IMDb test set.

Goals:

1. Load the cleaned IMDb test set.
2. Load the fitted TF-IDF vectorizer.
3. Load the trained PyTorch model.
4. Generate predictions for every test review.
5. Calculate final test metrics.
6. Analyze the model's mistakes.

Important:

The test set is used only for final evaluation.
We do not train the model or tune hyperparameters using test results.
In this notebook, we evaluate the saved TF-IDF + PyTorch
logistic-regression model on the official IMDb test set.

Goals:

1. Load the cleaned IMDb test set.
2. Load the fitted TF-IDF vectorizer.
3. Load the trained PyTorch model.
4. Generate predictions for every test review.
5. Calculate final test metrics.
6. Analyze the model's mistakes.

Important:

The test set is used only for final evaluation.
We do not train the model or tune hyperparameters using test results.


In [1]:
from __future__ import annotations

from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from datasets import load_from_disk

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

In [2]:
CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )

print("Current directory:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)

Current directory: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers\notebooks
Project root: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers


In [3]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "imdb_clean_splits"
)

MODEL_DIR = PROJECT_ROOT / "models"

REPORT_DIR = (
    PROJECT_ROOT
    / "reports"
    / "baseline"
    / "test_evaluation"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Data path:", DATA_PATH)
print("Model directory:", MODEL_DIR)
print("Report directory:", REPORT_DIR)

Data path: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers\data\processed\imdb_clean_splits
Model directory: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers\models
Report directory: D:\app\ai\Sentiment Analysis Traditional ML vs Transformers\reports\baseline\test_evaluation


## 3. Import reusable prediction code

Instead of defining the text-cleaning function and model architecture
again, we import them from `predict_tfidf.py`.

This keeps training, prediction, and evaluation consistent.

In [4]:
try:
    from src.predict_tfidf import (
        clean_text_for_model,
        load_artifacts,
    )
except ModuleNotFoundError:
    from predict_tfidf import (
        clean_text_for_model,
        load_artifacts,
    )

## 4. Load the test set and saved artifacts

In this section, we load:

- the official IMDb test set
- the fitted TF-IDF vectorizer
- the trained PyTorch model

No training happens in this notebook.

In [5]:
dataset = load_from_disk(
    str(DATA_PATH)
)

test_df = (
    dataset["test"]
    .to_pandas()[["text", "label"]]
    .copy()
)

label_names = (
    dataset["test"]
    .features["label"]
    .names
)

model, vectorizer, device = load_artifacts(
    model_dir=MODEL_DIR
)

print(f"Test rows: {len(test_df):,}")
print("Label names:", label_names)
print("Device:", device)
print(
    "Vectorizer features:",
    len(vectorizer.get_feature_names_out()),
)

Test rows: 25,000
Label names: ['neg', 'pos']
Device: cuda
Vectorizer features: 20000


## 5. Clean and vectorize the test set

We apply the same text-cleaning function used during training.

The saved vectorizer only transforms the test text.
It does not learn a new vocabulary or new IDF values.

In [6]:
test_texts = (
    test_df["text"]
    .map(clean_text_for_model)
    .tolist()
)

y_test = test_df["label"].to_numpy(
    dtype=np.int64
)

In [7]:
X_test = vectorizer.transform(
    test_texts
)

print("X_test shape:", X_test.shape)
print("X_test type:", type(X_test).__name__)
print("Non-zero values:", f"{X_test.nnz:,}")

X_test shape: (25000, 20000)
X_test type: csr_matrix
Non-zero values: 3,172,287


In [8]:
number_of_vectorizer_features = len(
    vectorizer.get_feature_names_out()
)

assert X_test.shape[0] == len(test_df)

assert X_test.shape[1] == (
    number_of_vectorizer_features
)

assert len(y_test) == len(test_df)

assert len(test_texts) == len(test_df)

print("All test-data checks passed.")

All test-data checks passed.


## 6. Run batched inference

The TF-IDF test matrix remains sparse.

We select a small group of rows, temporarily convert only that group
to a dense tensor, run the model, and repeat until every test review
has a prediction.

In [9]:
INFERENCE_BATCH_SIZE = 512

print(
    "Inference batch size:",
    INFERENCE_BATCH_SIZE,
)

Inference batch size: 512


In [11]:
def predict_in_batches(
    model: torch.nn.Module,
    sparse_features,
    device: torch.device,
    batch_size: int = 512,
) -> np.ndarray:
    """
    Predict positive-class probabilities for a sparse feature matrix.

    Only one batch is converted to a dense array at a time.
    """

    if batch_size <= 0:
        raise ValueError(
            "batch_size must be greater than zero."
        )

    number_of_examples = sparse_features.shape[0]

    probability_batches = []

    model.eval()

    with torch.inference_mode():
        for start_index in range(
            0,
            number_of_examples,
            batch_size,
        ):
            end_index = min(
                start_index + batch_size,
                number_of_examples,
            )

            sparse_batch = sparse_features[
                start_index:end_index
            ]

            dense_batch = (
                sparse_batch
                .toarray()
                .astype(
                    np.float32,
                    copy=False,
                )
            )

            feature_tensor = torch.from_numpy(
                dense_batch
            ).to(device)

            logits = model(
                feature_tensor
            )

            positive_probabilities = torch.sigmoid(
                logits
            )

            probability_batches.append(
                positive_probabilities
                .cpu()
                .numpy()
            )

    return np.concatenate(
        probability_batches
    )

### Generate test predictions

We measure the total inference time and convert the positive-class
probabilities into predicted class IDs using a threshold of 0.5.

In [12]:
inference_start_time = time.perf_counter()

test_positive_probabilities = predict_in_batches(
    model=model,
    sparse_features=X_test,
    device=device,
    batch_size=INFERENCE_BATCH_SIZE,
)

inference_seconds = (
    time.perf_counter()
    - inference_start_time
)

test_predictions = (
    test_positive_probabilities >= 0.5
).astype(np.int64)

test_negative_probabilities = (
    1.0 - test_positive_probabilities
)

print(
    "Inference time:",
    f"{inference_seconds:.2f} seconds",
)

print(
    "Number of predictions:",
    f"{len(test_predictions):,}",
)

Inference time: 1.31 seconds
Number of predictions: 25,000


In [13]:
assert test_positive_probabilities.shape == (
    len(test_df),
)

assert test_predictions.shape == (
    len(test_df),
)

assert np.all(
    test_positive_probabilities >= 0.0
)

assert np.all(
    test_positive_probabilities <= 1.0
)

assert set(
    np.unique(test_predictions)
).issubset({0, 1})

print("All inference checks passed.")

All inference checks passed.


In [14]:
prediction_preview = pd.DataFrame(
    {
        "actual_label_id": y_test[:10],
        "predicted_label_id": (
            test_predictions[:10]
        ),
        "positive_probability": (
            test_positive_probabilities[:10]
        ),
    }
)

prediction_preview[
    "actual_label"
] = prediction_preview[
    "actual_label_id"
].map(
    {
        0: label_names[0],
        1: label_names[1],
    }
)

prediction_preview[
    "predicted_label"
] = prediction_preview[
    "predicted_label_id"
].map(
    {
        0: label_names[0],
        1: label_names[1],
    }
)

prediction_preview

,actual_label_id,predicted_label_id,positive_probability,actual_label,predicted_label
0,0,0,0.308319,neg,neg
1,0,0,0.381565,neg,neg
2,0,0,0.347738,neg,neg
3,0,0,0.417665,neg,neg
4,0,1,0.502756,neg,pos
5,0,0,0.324278,neg,neg
6,0,0,0.495232,neg,neg
7,0,0,0.301296,neg,neg
8,0,0,0.247840,neg,neg
9,0,0,0.154898,neg,neg
